# Week 7 · Diffusion: generate images by denoising

Until now every network answered a question about its input: a number, a class,
a mask. A **generative model** produces new images that look as if they came
from the training set. A **diffusion model** gets there with a task you already
know, regression: it learns to predict the noise that was added to an image.
Sampling starts from pure noise and removes a little of it, many times.

![Forward process: a disk image is noised step by step until only noise is left. Reverse process: a network removes the noise step by step.](figures/diffusion-chain.png)

Build the four parts of a small diffusion model: the forward noising process,
the training loss, the timestep input of a U-Net, and the reverse sampling step.
Then diagnose and judge your generator. Assumes the Week 6 U-Net and PyTorch
training loops. Notation follows Lesson 7 of the course book.

Complete the **TODOs** and the written questions. Predict before running;
rerun cells in order after edits. Switch driver after activity 3.

Uses NumPy, Matplotlib, and CPU PyTorch. Data are generated locally; no GPU or
download needed. All training in this notebook takes about a minute.

In [ ]:
import math
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(2)
torch.manual_seed(7)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2})
BLUE, ORANGE, GREEN, GRAY = "#0072B2", "#D55E00", "#009E73", "#6b6b6b"  # colour-blind-safe
print("Ready: PyTorch", torch.__version__, "| CPU")

## 1. From predicting to generating

The training set holds 256 grayscale images of 16×16 pixels. Each shows one
smooth bright disk; its position and radius vary. Pixel values lie in $[-1, 1]$:
$-1$ is the dark background, $+1$ the bright centre of a disk. The images are
synthetic and small, so a network trains on them on a laptop in under a minute.

A generator should produce **new** disks: plausible positions and sizes, but not
copies of training images.

In [ ]:
# Supplied: one smooth bright disk per image. Position and radius vary.
def make_disks(n, seed):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:16, :16]
    images = np.zeros((n, 1, 16, 16), dtype=np.float32)
    for i in range(n):
        radius = rng.uniform(2.0, 4.0)
        cy, cx = rng.uniform(radius + 1, 14 - radius, size=2)  # the whole disk stays inside
        distance = np.sqrt((yy - cy)**2 + (xx - cx)**2)
        images[i, 0] = 1 / (1 + np.exp((distance - radius) / 0.6))  # soft edge, values in (0, 1)
    return torch.tensor(2 * images - 1)  # rescale brightness from [0, 1] to [-1, 1]

# Supplied: show rows of images. `rows` is a list of tensors with shape (K, 1, 16, 16).
def show_rows(rows, labels=None, titles=None, suptitle=None, size=1.0):
    columns = max(len(row) for row in rows)
    width = size * columns + (1.1 if labels else 0.2)
    fig, axes = plt.subplots(len(rows), columns, squeeze=False,
                             figsize=(width, 1.08 * size * len(rows) + 0.5))
    for ax in axes.flat:
        ax.set(xticks=[], yticks=[])
        ax.spines[:].set_visible(False)
    for r, images in enumerate(rows):
        for c, image in enumerate(images):
            axes[r, c].imshow(image[0].detach().numpy(), cmap="gray", vmin=-1, vmax=1,
                              interpolation="nearest")
        if labels:
            axes[r, 0].set_ylabel(labels[r], rotation=0, ha="right", va="center", fontsize=9)
    for ax, title in zip(axes[0], titles or []):
        ax.set_title(title, fontsize=9)
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    fig.tight_layout()
    plt.show()

train = make_disks(256, seed=7)  # used for training
valid = make_disks(64, seed=8)   # independent images, never used for training
print(f"Training images: {tuple(train.shape)} | "
      f"values from {train.min().item():.2f} to {train.max().item():.2f}")
show_rows([train[:8], train[8:16]], suptitle="Sixteen training images")

### A tempting shortcut: treat every pixel separately

For each of the 256 pixel positions, copy the value from a randomly chosen
training image. Every single pixel then follows exactly the distribution it has
in the training set.

**Predict:** will the result look like a disk? Sketch what you expect.

In [ ]:
generator = torch.Generator().manual_seed(1)
# For every pixel of 8 new images, choose a random training image to copy that pixel from.
donor = torch.randint(len(train), (8, 16, 16), generator=generator)
row, column = torch.meshgrid(torch.arange(16), torch.arange(16), indexing="ij")
pixel_soup = train[donor, 0, row, column].unsqueeze(1)  # shape (8, 1, 16, 16)
show_rows([train[:8], pixel_soup], labels=["Training", "Independent\npixels"])

**Explain:** every pixel has the right distribution, yet the images are wrong.
What does this baseline ignore? What must a generator know about neighbouring
pixels?

_Your prediction and explanation:_ …

**The plan.** Producing a whole image in one step is hard. A diffusion model
splits the work into many easy steps:

1. **Forward process:** destroy training images gradually with noise. Fixed, nothing to learn.
2. **Denoising:** train a network to tell which part of a noisy image is noise.
3. **Reverse process:** start from pure noise and remove a little noise, again and again.

## 2. The forward process: destroy an image on purpose

Choose a number of steps $T$ and a **noise schedule** $\beta_1,\dots,\beta_T$
with $0<\beta_t<1$. Step $t$ shrinks the image slightly and adds a little
Gaussian noise:

$$x_t=\sqrt{1-\beta_t}\;x_{t-1}+\sqrt{\beta_t}\;z_t,\qquad z_t\sim\mathcal N(0,I).$$

$x_0$ is the clean image. Define $\alpha_t=1-\beta_t$ and the **signal level**
$\bar\alpha_t=\alpha_1\alpha_2\cdots\alpha_t$, with $\bar\alpha_0=1$. Then all
$t$ steps collapse into one **direct noising equation**:

$$\boxed{\,x_t=\sqrt{\bar\alpha_t}\;x_0+\sqrt{1-\bar\alpha_t}\;\epsilon,\qquad \epsilon\sim\mathcal N(0,I)\,}$$

The squared weights add up to 1: signal is traded for noise.

**Pen and paper:** take a single pixel with $x_0=1$ and two steps with
$\beta_1=\beta_2=0.19$, so that $\sqrt{1-\beta_t}=0.9$.

1. Write $x_2$ in terms of $x_0$, $z_1$ and $z_2$.
2. Give the mean and the variance of $x_2$. Independent noise terms add their
   variances; a factor $a$ multiplies a variance by $a^2$.
3. Calculate $\bar\alpha_2$. Do $\sqrt{\bar\alpha_2}$ and $1-\bar\alpha_2$ match
   your mean and variance?

_Your calculations:_ …

In [ ]:
# Supplied. Index 0 is a placeholder, so schedule.abar[t] matches the maths for t = 0, ..., T.
def make_schedule(beta):
    alpha = 1 - beta
    abar = torch.cumprod(alpha, dim=0)  # abar[0] = 1
    return SimpleNamespace(steps=len(beta) - 1, beta=beta.float(), alpha=alpha.float(),
                           abar=abar.float())

def cosine_schedule(steps, offset=0.008):
    t = torch.arange(steps + 1, dtype=torch.float64)
    # f is the desired signal level abar[t], up to a constant factor.
    f = torch.cos((t / steps + offset) / (1 + offset) * math.pi / 2)**2
    beta = torch.zeros(steps + 1, dtype=torch.float64)
    beta[1:] = (1 - f[1:] / f[:-1]).clamp(max=0.999)  # beta[t] = 1 - abar[t] / abar[t-1]
    return make_schedule(beta)

T = 64
schedule = cosine_schedule(T)
print("Signal level abar[t] at t = 0, 16, 32, 48, 64:",
      "  ".join(f"{schedule.abar[t].item():.4f}" for t in (0, 16, 32, 48, 64)))

In [ ]:
def q_sample(clean, t, noise, schedule):
    # Jump from clean images x_0 directly to x_t. `t` holds one timestep per image.
    abar = schedule.abar[t].view(-1, 1, 1, 1)  # shape (N, 1, 1, 1): one signal level per image
    # TODO 1: apply the direct noising equation. `abar` broadcasts over channel, height and width.
    return None

probe_t = torch.tensor([1, 32, 64])
ones, zeros = torch.ones(3, 1, 16, 16), torch.zeros(3, 1, 16, 16)
forward_ready = q_sample(ones, probe_t, zeros, schedule) is not None
if forward_ready:
    signal_only = q_sample(ones, probe_t, zeros, schedule)  # no noise: the signal weight remains
    noise_only = q_sample(zeros, probe_t, ones, schedule)   # no signal: the noise weight remains
    assert signal_only.shape == ones.shape
    torch.testing.assert_close(signal_only[:, 0, 0, 0], schedule.abar[probe_t].sqrt())
    torch.testing.assert_close(noise_only[:, 0, 0, 0], (1 - schedule.abar[probe_t]).sqrt())
    print("Direct noising checked: signal and noise weights are correct at t = 1, 32, 64.")
else:
    print("Complete TODO 1, then rerun this cell.")

**Check the shortcut against the slow way.** The next cell noises 10,000 copies
of one training image twice: with 32 single steps, and with one direct jump to
$t=32$. It reports the mean and the variance of the disk's brightest pixel,
whose clean value is 0.985.

**Before running:** use $\bar\alpha_{32}=0.4938$ from the output above to
calculate the mean and the variance that the direct noising equation predicts.

_Your calculation:_ …

In [ ]:
if forward_ready:
    generator = torch.Generator().manual_seed(2)
    copies = train[:1].repeat(10_000, 1, 1, 1)
    x = copies.clone()
    for step in range(1, 33):  # 32 single steps, one after the other
        z = torch.randn(x.shape, generator=generator)
        x = (1 - schedule.beta[step]).sqrt() * x + schedule.beta[step].sqrt() * z
    noise = torch.randn(x.shape, generator=generator)
    jump = q_sample(copies, torch.full((len(copies),), 32), noise, schedule)
    brightest = train[0].flatten().argmax()
    print(f"Clean value of the brightest pixel: {train[0].flatten()[brightest].item():.3f}")
    for name, result in [("32 single steps", x), ("One direct jump", jump)]:
        values = result.flatten(1)[:, brightest]
        print(f"{name}: mean {values.mean().item():.3f}, variance {values.var().item():.3f}")
else:
    print("Complete TODO 1 first.")

### Choose a schedule that really ends in noise

Sampling will start from pure Gaussian noise, so the forward process must end
there: $\bar\alpha_T\approx0$.

The original DDPM paper used $T=1000$ steps with $\beta_t$ rising linearly from
$0.0001$ to $0.02$. We use only $T=64$ steps to keep sampling fast.

**Predict:** suppose we copy the same range of $\beta_t$ values for our 64 steps.
Will $x_{64}$ be pure noise? The cell compares this *copied linear* schedule with
the **cosine schedule** used in the rest of the notebook. Both image rows use the
same noise $\epsilon$.

In [ ]:
copied_beta = torch.zeros(T + 1, dtype=torch.float64)
copied_beta[1:] = torch.linspace(1e-4, 0.02, T, dtype=torch.float64)  # the DDPM range for T = 1000
copied_linear = make_schedule(copied_beta)

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.plot(schedule.abar.numpy(), color=BLUE, label="Cosine")
ax.plot(copied_linear.abar.numpy(), color=ORANGE, label="Copied linear")
ax.set(xlabel="Timestep $t$", ylabel=r"Signal level $\bar\alpha_t$", xlim=(0, T), ylim=(0, 1.02))
ax.grid(color="#e6e6e6", linewidth=0.8)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
print(f"Signal level at t = 64: cosine {schedule.abar[T].item():.1e} | "
      f"copied linear {copied_linear.abar[T].item():.2f}")

if forward_ready:
    shown = torch.tensor([0, 8, 16, 32, 48, 64])
    clean = valid[:1].repeat(len(shown), 1, 1, 1)
    noise = torch.randn(1, 1, 16, 16, generator=torch.Generator().manual_seed(3))
    noise = noise.repeat(len(shown), 1, 1, 1)  # the same noise at every timestep
    show_rows([q_sample(clean, shown, noise, schedule),
               q_sample(clean, shown, noise, copied_linear)],
              labels=["Cosine", "Copied\nlinear"], titles=[f"t = {t}" for t in shown.tolist()])

**Inspect and explain:**

1. How much of the clean image is left in $x_{64}$ under each schedule? The
   image is multiplied by $\sqrt{\bar\alpha_{64}}$.
2. With the cosine schedule, at which $t$ can you no longer see the disk? Is
   the information about the disk gone at that point?
3. Sampling starts from pure noise. Why would a network trained with the copied
   linear schedule meet inputs at the start of sampling that it never saw in
   training?

_Your observations:_ …

## 3. Learning to denoise

### A regression task whose targets come for free

Take a clean training image $x_0$. Draw a timestep $t$ uniformly from
$\{1,\dots,T\}$ and noise $\epsilon\sim\mathcal N(0,I)$, and form $x_t$ with the
direct noising equation. We made $\epsilon$ ourselves, so we know it. A network
$\epsilon_\theta(x_t,t)$ sees the noisy image and the timestep, and is trained
to predict that noise:

$$L_{\text{simple}}(\theta)=\mathbb E_{x_0,\,t,\,\epsilon}\Big[\tfrac1D\,\lVert\epsilon-\epsilon_\theta(x_t,t)\rVert_2^2\Big],\qquad D=256\text{ pixels}.$$

This is the mean squared error from Week 1, with one target value per pixel.

### Noise and clean image are two views of one prediction

Whoever knows the noise in $x_t$ also knows the clean image. Solve the direct
noising equation for $x_0$ and insert the predicted noise:

$$\hat x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\;\epsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}.$$

**Predict** the loss of two "denoisers" that need no training:

1. **Always zero** claims there is no noise: $\epsilon_\theta=0$. Which number
   do you expect, given that every noise value has mean 0 and variance 1?
2. **Copy the input** claims everything is noise: $\epsilon_\theta=x_t$. At which
   noise levels is this a good guess, at which a bad one?

_Your predictions:_ …

In [ ]:
def predict_clean(noisy, t, noise, schedule):
    # Turn a noise prediction for x_t into an estimate of the clean image x_0.
    abar = schedule.abar[t].view(-1, 1, 1, 1)
    # TODO 2: solve the direct noising equation for the clean image.
    return None

def diffusion_loss(model, clean, schedule, generator):
    # Noise-prediction loss for one minibatch of clean images.
    # Supplied: one random timestep per image, and noise with the shape of the images.
    t = torch.randint(1, schedule.steps + 1, (len(clean),), generator=generator)
    noise = torch.randn(clean.shape, generator=generator)
    # TODO 3a: noise every image to its own level t. Reuse q_sample.
    noisy = None
    if noisy is None:
        return None
    prediction = model(noisy, t)
    # TODO 3b: return the mean squared error between the predicted and the true noise.
    return None

# Supplied reference "denoisers" without learned weights.
class ZeroNoise(nn.Module):  # "there is no noise"
    def forward(self, x, t):
        return torch.zeros_like(x)

class CopyInput(nn.Module):  # "everything is noise"
    def forward(self, x, t):
        return x

class Oracle(nn.Module):  # knows the clean images and returns the exact noise
    def __init__(self, clean, schedule):
        super().__init__()
        self.clean, self.schedule = clean, schedule

    def forward(self, x, t):
        abar = self.schedule.abar[t].view(-1, 1, 1, 1)
        return (x - abar.sqrt() * self.clean) / (1 - abar).sqrt()

clean_ready = loss_ready = False
if forward_ready:
    check_t = torch.tensor([1, 16, 32, 48])
    check_noise = torch.randn(4, 1, 16, 16, generator=torch.Generator().manual_seed(3))
    check_noisy = q_sample(train[:4], check_t, check_noise, schedule)
    estimate = predict_clean(check_noisy, check_t, check_noise, schedule)
    clean_ready = estimate is not None
    if clean_ready:
        torch.testing.assert_close(estimate, train[:4], atol=1e-4, rtol=0)
        print("Clean estimate checked: the true noise leads back to the clean image.")
    zero_loss = diffusion_loss(ZeroNoise(), train, schedule, torch.Generator().manual_seed(4))
    loss_ready = zero_loss is not None
    if loss_ready:
        assert zero_loss.ndim == 0, "Return one number: the mean over all images and pixels."
        assert abs(zero_loss.item() - 1) < 0.05, "Always predicting zero should cost about 1."
        exact = Oracle(train, schedule)
        exact_loss = diffusion_loss(exact, train, schedule, torch.Generator().manual_seed(4))
        assert exact_loss.item() < 1e-6, "Predicting the exact noise should cost 0."
        print(f"Loss checked: always zero {zero_loss.item():.3f} | "
              f"exact noise {exact_loss.item():.1e}")
if not (forward_ready and clean_ready and loss_ready):
    print("Complete TODOs 1 to 3, then rerun this cell.")

**Discuss:**

1. The training set has no labels. Where do the regression targets come from?
   Why can we create as many different training pairs as we like from only 256 images?
2. Every image in a minibatch gets its own random $t$. What would we lose by
   training one separate network for each noise level instead?

_Your answers:_ …

## 4. A U-Net that knows the noise level

The denoiser maps a noisy 16×16 image to a 16×16 noise prediction. An
encoder–decoder with skip connections suits this task: the bottleneck sees the
whole image and can work out *where the disk is*; the skips keep the detail
needed to predict noise *pixel by pixel*. Compared with Week 6:

| | Week 6 segmentation U-Net | Denoising U-Net |
|---|---|---|
| Input | image | noisy image $x_t$ **and timestep $t$** |
| Output | one logit per pixel | one noise value per pixel, any real number |
| Loss | binary cross-entropy against a mask | squared error against the noise we added |
| Block | convolution → ReLU | convolution → GroupNorm → SiLU, residual shortcut, **time shift** |

GroupNorm normalises every image on its own, so the result does not depend on
the other noise levels in the minibatch. SiLU, $x\cdot\mathrm{sigmoid}(x)$, is a
smooth relative of ReLU.

### Tell the network the timestep

One network serves all noise levels. A nearly clean image ($t=2$) and almost
pure noise ($t=62$) call for very different outputs, so the network receives
$t$ as a second input. The timestep is scaled to $\tau=t/T$, expanded into five
features $(\tau,\ \sin\pi\tau,\ \cos\pi\tau,\ \sin2\pi\tau,\ \cos2\pi\tau)$, and passed
through a small MLP. The result is the **time embedding** $e(t)\in\mathbb R^{32}$.

### Inject time into every block

Each block owns a linear layer that turns $e(t)$ into **one number per
channel**. The block adds this number to every pixel of that channel's feature
map: a bias that depends on the noise level.

**Before coding:** inside a block the feature maps `h` have shape `(N, C, H, W)`
and `self.time(emb)` has shape `(N, C)`. Which shape must the shift have so that
PyTorch broadcasts it over height and width? What happens if you add the
`(N, C)` tensor directly?

_Your answer:_ …

In [ ]:
def time_features(t, steps):
    tau = t.float() / steps
    features = [tau, torch.sin(math.pi * tau), torch.cos(math.pi * tau),
                torch.sin(2 * math.pi * tau), torch.cos(2 * math.pi * tau)]
    return torch.stack(features, dim=1)  # (N, 5)

class TimeBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(4, out_channels)
        self.time = nn.Linear(32, out_channels)  # time embedding -> one shift per channel
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(4, out_channels)
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x, emb):
        h = self.norm1(self.conv1(x))  # (N, C, H, W)
        shift = self.time(emb)         # (N, C)
        # TODO 4: add the shift to h so that every pixel of a channel receives that channel's value.
        h = h
        h = F.silu(h)
        h = self.norm2(self.conv2(h))
        return F.silu(h + self.shortcut(x))  # residual shortcut, as in Week 5

class TinyTimeUNet(nn.Module):
    def __init__(self, steps, width=8):
        super().__init__()
        self.steps = steps
        self.embedding = nn.Sequential(nn.Linear(5, 32), nn.SiLU(), nn.Linear(32, 32))
        self.enc1 = TimeBlock(1, width)
        self.enc2 = TimeBlock(width, 2 * width)
        self.bottom = TimeBlock(2 * width, 4 * width)
        self.dec2 = TimeBlock(4 * width + 2 * width, 2 * width)
        self.dec1 = TimeBlock(2 * width + width, width)
        self.output = nn.Conv2d(width, 1, kernel_size=1)

    def forward(self, x, t):
        emb = self.embedding(time_features(t, self.steps))    # (N, 32)
        skip1 = self.enc1(x, emb)                             # 8 × 16 × 16
        skip2 = self.enc2(F.avg_pool2d(skip1, 2), emb)        # 16 × 8 × 8
        h = self.bottom(F.avg_pool2d(skip2, 2), emb)          # 32 × 4 × 4
        h = F.interpolate(h, scale_factor=2, mode="nearest")
        h = self.dec2(torch.cat([h, skip2], dim=1), emb)      # 16 × 8 × 8
        h = F.interpolate(h, scale_factor=2, mode="nearest")
        h = self.dec1(torch.cat([h, skip1], dim=1), emb)      # 8 × 16 × 16
        return self.output(h)                                 # predicted noise: no sigmoid

torch.manual_seed(17)
model = TinyTimeUNet(T)
probe = torch.randn(4, 1, 16, 16, generator=torch.Generator().manual_seed(5))
nearly_clean, nearly_noise = torch.full((4,), 2), torch.full((4,), 62)
output = model(probe, nearly_clean)
assert output.shape == probe.shape, "The network must return one value per input pixel."
output.sum().backward()
time_gradient = model.embedding[0].weight.grad  # None if the timestep never reaches the output
time_ready = (time_gradient is not None and time_gradient.abs().sum().item() > 0
              and not torch.allclose(output, model(probe, nearly_noise)))
print("Parameters:", sum(p.numel() for p in model.parameters()))
if time_ready:
    print("Time input checked: the same image gives different outputs at t = 2 and t = 62.")
else:
    print("The network ignores the timestep. Complete TODO 4, then rerun this cell.")

**Check your understanding:**

1. The output layer has no sigmoid. Why would a sigmoid be wrong for this task?
2. Follow `emb` through `TinyTimeUNet.forward`. How many blocks receive the
   time embedding? Why might telling only the first block be too little?

_Your answers:_ …

## 5. Train across all noise levels

One update: draw 32 training images, give each its own random timestep and
noise, predict the noise, and take an Adam step on your `diffusion_loss`. Start
with a **short run of 200 updates**.

To see what the network has learned, the cell measures the validation error
**separately at every noise level** and compares it with the two baselines
from activity 3.

**Predict before running:** at which noise levels will the network have its
lowest error, and at which its highest?

_Your prediction:_ …

In [ ]:
# Supplied training and evaluation helpers.
def train_updates(model, optimizer, images, schedule, updates, generator):
    model.train()
    losses = []
    for _ in range(updates):
        batch = images[torch.randint(len(images), (32,), generator=generator)]  # 32 random images
        loss = diffusion_loss(model, batch, schedule, generator)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

@torch.no_grad()
def denoising_errors(model, images, schedule):
    # Mean squared noise error at every timestep 1, ..., T.
    # Every model is tested with the same noise.
    model.eval()
    noise = torch.randn(images.shape, generator=torch.Generator().manual_seed(0))
    errors = []
    for step in range(1, schedule.steps + 1):
        t = torch.full((len(images),), step)
        prediction = model(q_sample(images, t, noise, schedule), t)
        errors.append(F.mse_loss(prediction, noise).item())
    return np.array(errors)

def plot_errors(errors):
    fig, ax = plt.subplots(figsize=(6, 3.2))
    for name, color in zip(errors, [GRAY, ORANGE, BLUE, GREEN]):
        ax.plot(np.arange(1, T + 1), errors[name], color=color, label=name)
    ax.set(xlabel="Timestep $t$   (almost clean → almost pure noise)",
           ylabel="Validation noise error (log scale)", yscale="log", xlim=(1, T), ylim=(5e-4, 3))
    ax.grid(color="#e6e6e6", linewidth=0.8)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    plt.show()

report_steps = [1, 16, 32, 48, 64]

def print_errors(errors):
    print(f"{'Validation noise error at t =':30s}" + "".join(f"{t:>9d}" for t in report_steps))
    for name, values in errors.items():
        print(f"{name:30s}" + "".join(f"{values[t - 1]:9.4f}" for t in report_steps))

losses = None
ready_to_train = forward_ready and clean_ready and loss_ready and time_ready
if ready_to_train:
    torch.manual_seed(17)  # the same initial weights on every run of this cell
    model = TinyTimeUNet(T)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
    train_generator = torch.Generator().manual_seed(19)
    losses = train_updates(model, optimizer, train, schedule, 200, train_generator)

    fig, ax = plt.subplots(figsize=(6, 2.6))
    ax.plot(np.arange(1, len(losses) + 1), losses, color=BLUE, linewidth=1)
    ax.set(xlabel="Update", ylabel="Training loss (log scale)", yscale="log", xlim=(1, len(losses)))
    ax.grid(color="#e6e6e6", linewidth=0.8)
    fig.tight_layout()
    plt.show()

    errors = {"Always zero": denoising_errors(ZeroNoise(), valid, schedule),
              "Copy the input": denoising_errors(CopyInput(), valid, schedule),
              f"Network, {len(losses)} updates": denoising_errors(model, valid, schedule)}
    plot_errors(errors)
    print_errors(errors)
    factor = ((1 - schedule.abar) / schedule.abar).sqrt()
    print(f"{'Noise error → image error: ×':30s}"
          + "".join(f"{factor[t].item():9.3f}" for t in report_steps))
else:
    print("Complete TODOs 1 to 4, then rerun in order.")

The last printed row is the factor $\sqrt{1-\bar\alpha_t}\,/\sqrt{\bar\alpha_t}$:
an error of size $\delta$ in the predicted noise changes the clean estimate
$\hat x_0$ by $\delta$ times this factor.

**Stop and interpret:**

1. Does the network beat *both* baselines at every noise level? Look closely
   at the right-hand end of the plot.
2. The network's error is smallest near $t=64$. Is that an achievement? What
   does *copy the input* reach there, and why?
3. The error is largest near $t=1$, where the image is almost clean. Use the
   factor to compare: at which end of the chain do small noise errors become
   large errors in the clean estimate?
4. The training loss jumps up and down far more than in Week 6. Name one
   source of randomness in every update that Week 6 did not have.

_Your observations:_ …

## 6. The reverse process: generate in small steps

Sampling runs the chain backwards. Start from pure noise
$x_T\sim\mathcal N(0,I)$ and go from $x_t$ to $x_{t-1}$ until you reach $x_0$.

**If we knew the clean image**, the previous state would be easy to draw. Given
$x_t$ and $x_0$, it is Gaussian with mean $\tilde\mu_t$ and variance $\tilde\beta_t$:

$$\tilde\mu_t(x_t,x_0)=\underbrace{\frac{\beta_t\sqrt{\bar\alpha_{t-1}}}{1-\bar\alpha_t}}_{\text{clean weight}}\;x_0\;+\;\underbrace{\frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}}_{\text{state weight}}\;x_t,\qquad \tilde\beta_t=\beta_t\,\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}.$$

**We do not know the clean image**, so the network supplies its current
estimate $\hat x_0$ from activity 3, clipped to the valid range $[-1,1]$. One
**DDPM step** is

$$x_{t-1}=\tilde\mu_t(x_t,\hat x_0)+\sqrt{\tilde\beta_t}\;z,\qquad z\sim\mathcal N(0,I).$$

The step mixes the clean estimate with the current state and adds **fresh
noise**. Then the network looks again, at a slightly lower noise level.

**Pen and paper:** evaluate the clean weight, the state weight and
$\tilde\beta_t$ for the last step, $t=1$. Use $\bar\alpha_0=1$ and
$\bar\alpha_1=\alpha_1=1-\beta_1$. What does the last step return? Does it add
noise?

_Your calculation:_ …

In [ ]:
# Supplied: the three reverse-step tables, indexed by t like schedule.abar. Index 0 is unused.
beta, alpha, abar = schedule.beta[1:], schedule.alpha[1:], schedule.abar[1:]
previous = schedule.abar[:-1]  # abar[t-1] for t = 1, ..., T; begins with abar[0] = 1
unused = torch.zeros(1)
schedule.clean_weight = torch.cat([unused, beta * previous.sqrt() / (1 - abar)])
schedule.state_weight = torch.cat([unused, alpha.sqrt() * (1 - previous) / (1 - abar)])
schedule.variance = torch.cat([unused, beta * (1 - previous) / (1 - abar)])

fig, ax = plt.subplots(figsize=(6, 3))
steps = np.arange(1, T + 1)
ax.plot(steps, schedule.state_weight[1:].numpy(), color=BLUE, label="State weight")
ax.plot(steps, schedule.clean_weight[1:].numpy(), color=ORANGE, label="Clean weight")
ax.plot(steps, schedule.variance[1:].sqrt().numpy(), color=GREEN,
        label=r"Fresh noise $\sqrt{\tilde\beta_t}$")
ax.set(xlabel="Timestep $t$   (sampling runs from right to left)",
       ylabel="Weight or standard deviation", xlim=(1, T), ylim=(0, 1.05))
ax.grid(color="#e6e6e6", linewidth=0.8)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()
for t in (64, 32, 1):
    print(f"t = {t:2d}: clean weight {schedule.clean_weight[t].item():.3f} | "
          f"state weight {schedule.state_weight[t].item():.3f} | "
          f"fresh noise {schedule.variance[t].sqrt().item():.3f}")

**Read the plot** and compare the printed values at $t=1$ with your calculation.
During most of the chain, is the new state closer to the old state or to the
clean estimate? Where does the clean estimate take over?

_Your reading:_ …

In [ ]:
@torch.no_grad()
def ddpm_step(model, x, step, noise, schedule):
    # One reverse step from x_t (`x`) to x_{t-1}. `step` is the integer t, `noise` is z.
    t = torch.full((len(x),), step)
    clean = predict_clean(x, t, model(x, t), schedule).clamp(-1, 1)  # clipped estimate of x_0
    # TODO 5a: mean of the previous state. Combine `clean` and `x` with
    # schedule.clean_weight[step] and schedule.state_weight[step].
    mean = None
    if mean is None:
        return None
    # TODO 5b: return the mean plus `noise` with standard deviation sqrt(schedule.variance[step]).
    return None

# Supplied: run all reverse steps T, ..., 1.
# Returns the results and the states x_t for the timesteps listed in `record`.
@torch.no_grad()
def sample_ddpm(model, initial, schedule, seed, fresh_noise=True, record=()):
    model.eval()
    generator = torch.Generator().manual_seed(seed)
    x, states = initial.clone(), {}
    for step in range(schedule.steps, 0, -1):
        if step in record:
            states[step] = x.clone()
        z = torch.randn(x.shape, generator=generator)
        x = ddpm_step(model, x, step, z if fresh_noise else torch.zeros_like(z), schedule)
    return x, states

sampler_ready = False
if forward_ready and clean_ready:
    known = valid[:1]  # the oracle knows this clean image and returns the exact noise in x_t
    oracle = Oracle(known, schedule)
    starts = torch.randn(8, 1, 16, 16, generator=torch.Generator().manual_seed(6))
    no_noise = torch.zeros_like(starts)
    sampler_ready = ddpm_step(oracle, starts, T, no_noise, schedule) is not None
    if sampler_ready:
        # 1. One step without fresh noise must match an equivalent formula for the same mean.
        state = q_sample(known.repeat(8, 1, 1, 1), torch.full((8,), 32), starts, schedule)
        same_mean = state - schedule.beta[32] / (1 - schedule.abar[32]).sqrt() * starts
        same_mean = same_mean / schedule.alpha[32].sqrt()
        step_mean = ddpm_step(oracle, state, 32, no_noise, schedule)
        mean_ok = torch.allclose(step_mean, same_mean, atol=1e-5)
        assert mean_ok, "TODO 5a: the mean is wrong. Check which weight multiplies which image."
        # 2. Fresh noise enters with standard deviation sqrt(variance): none at all at t = 1.
        for step in (32, 1):
            with_noise = ddpm_step(oracle, starts, step, torch.ones_like(starts), schedule)
            without = ddpm_step(oracle, starts, step, no_noise, schedule)
            expected = torch.full_like(starts, schedule.variance[step].sqrt().item())
            noise_ok = torch.allclose(with_noise - without, expected, atol=1e-5)
            assert noise_ok, "TODO 5b: the noise needs standard deviation sqrt(variance[step])."
        # 3. With exact noise predictions, every start must end at the known image.
        result, _ = sample_ddpm(oracle, starts, schedule, seed=7)
        deviation = (result - known).abs().max().item()
        assert deviation < 1e-3, f"All starts must end at the known image (off by {deviation:.2f})."
        print("Sampler checked: 8 different starts all end at the known image "
              f"(largest deviation {deviation:.0e}).")
if not sampler_ready:
    print("Complete TODOs 1, 2 and 5, then rerun this cell.")

**Explain:** the oracle test replaces the network with a perfect denoiser for
one known image. What does a passing test tell you about your sampler? What can
it *not* tell you about your trained network?

_Your answer:_ …

### Generate

Draw 64 starting-noise images and run all 64 reverse steps with the network from
the short training run. The first figure shows 32 results, numbered row by row
from 0. The second follows the samples listed in `watch` through the chain.

In [ ]:
# Supplied: for the samples in `watch`, show the states x_t and the network's clean estimates.
@torch.no_grad()
def show_sampling(model, states, results, watch):
    rows, labels = [], []
    for i in watch:
        estimates = []
        for step, x in states.items():
            t = torch.full((1,), step)
            estimate = predict_clean(x[i:i + 1], t, model(x[i:i + 1], t), schedule)
            estimates.append(estimate.clamp(-1, 1))
        rows.append(torch.cat([x[i:i + 1] for x in states.values()] + [results[i:i + 1]]))
        rows.append(torch.cat(estimates))
        labels += [f"Sample {i}\nstate", "clean\nestimate"]
    show_rows(rows, labels=labels, titles=[f"t = {step}" for step in states] + ["result"])

first_samples = None
if losses is not None and sampler_ready:
    # 64 starting-noise images. Every later experiment reuses them.
    start = torch.randn(64, 1, 16, 16, generator=torch.Generator().manual_seed(37))
    shown_steps = [64, 48, 32, 16, 8, 1]
    first_samples, first_states = sample_ddpm(model, start, schedule, seed=31, record=shown_steps)
    first_updates = len(losses)
    row_labels = ["0–7", "8–15", "16–23", "24–31"]
    show_rows([first_samples[i:i + 8] for i in range(0, 32, 8)], labels=row_labels,
              suptitle=f"Generated after {first_updates} updates", size=0.8)
    watch = [0, 1, 2]  # change these numbers to follow other samples
    show_sampling(model, first_states, first_samples, watch)
else:
    print("Train the network and complete the sampler first, then rerun in order.")

**Inspect** the second figure. For each sample, the upper row shows the state
$x_t$ that the network receives; the lower row shows its clean estimate $\hat x_0$.

1. At which step can you first tell where the disk will be? Do you see it
   earlier in the state or in the clean estimate?
2. How does the clean estimate change from $t=48$ to $t=1$? Why is an early
   estimate blurry? Think of all the clean images that could hide in such a
   noisy state.
3. The estimate at $t=64$ is saturated speckle. Explain why with the factor
   printed in activity 5. Then use the weights at $t=64$ to explain why this
   does not ruin the sample.

_Your observations:_ …

### Diagnose before you tune: sampler or training?

Look at the 32 images from the short run and count those that are **empty**,
**faint**, or **not round**. Your counts will differ from your neighbours':
tiny numerical differences between computers grow during training.

There are two suspects: a mistake in the **sampler**, or a **network** that is
not trained well enough.

1. Which evidence do you already have about each suspect?
2. Put the number of one failed image into `watch` and rerun the cell above.
   At which step does the sample go wrong?

_Your diagnosis:_ …

Now test your diagnosis. The next cell trains for **1600 more updates** (1800
in total) and samples again with **the same random numbers**: the same starting
noise and the same fresh noise.

In [ ]:
samples = None
if first_samples is not None:
    if len(losses) == first_updates:  # rerunning this cell does not train again
        losses += train_updates(model, optimizer, train, schedule, 1600, train_generator)
    errors[f"Network, {len(losses)} updates"] = denoising_errors(model, valid, schedule)
    samples, states = sample_ddpm(model, start, schedule, seed=31, record=shown_steps)
    plot_errors(errors)
    print_errors(errors)
    for images, updates in [(first_samples, first_updates), (samples, len(losses))]:
        show_rows([images[i:i + 8] for i in range(0, 32, 8)], labels=row_labels,
                  suptitle=f"Generated after {updates} updates", size=0.8)
    # Optional: follow the same samples again with the better network.
    # show_sampling(model, states, samples, watch)
else:
    print("Generate images with the short run first, then rerun in order.")

**Record and explain:**

| Network | Empty or faint (of 32) | Not round (of 32) | Error at $t=1$ | Error at $t=32$ | Error at $t=64$ |
|---|---|---|---|---|---|
| 200 updates | … | … | … | … | … |
| 1800 updates | … | … | … | … | … |

1. Was your diagnosis right? Which observation made *before* the longer
   training already pointed to the network and not to the sampler?
2. Both figures use the same random numbers. Compare them image by image. What
   stays the same, what changes? What does the noise decide, and what does the
   network decide?
3. You have now turned one dial, **training quality**, while the sampler stayed
   fixed. Name a second dial that changes the *cost* of generating an image
   without retraining. (Lab 7 turns it.)

_Your observations:_ …

### What is the fresh noise for?

**Predict:** delete the fresh noise from every reverse step, so that each step
returns only the mean $\tilde\mu_t$. The 64 starting-noise images are still all
different. What will the 64 results look like?

_Your prediction:_ …

In [ ]:
if samples is not None:
    without_noise, _ = sample_ddpm(model, start, schedule, seed=31, fresh_noise=False)
    show_rows([samples[:8], without_noise[:8]], labels=["With fresh\nnoise", "Without"])
    for name, images in [("With fresh noise", samples), ("Without fresh noise", without_noise)]:
        spread = images.std(dim=0).mean().item()  # how much a pixel varies across the 64 images
        print(f"{name}: pixel standard deviation across the 64 images {spread:.3f}")
else:
    print("Train the network for the full 1800 updates first, then rerun in order.")

**Explain:** every result looks like a clean disk. Why is this sampler still a
poor generator? Where does the variety of DDPM samples come from? Look again at
the state weight and the fresh noise at $t=64$ in the plot above.

_Your explanation:_ …

## 7. Judge the generator

A generated image has no ground truth: there is no "correct" disk for a given
noise input. So we judge the **set** of samples. A good set has **fidelity**
(every sample looks like a real disk) and **diversity** (together the samples
cover the variety of the data), and it does not simply repeat the training set.

Two simple checks:

- **Copies?** For every generated image, find the nearest training image by
  Euclidean distance over the 256 pixels. As a yardstick, do the same for the
  validation images: they are new disks from the true distribution.
- **Same variety?** Compare the **bright area**, the fraction of pixels above 0
  and a stand-in for disk size, between generated and validation images.

**Predict:** if the network had memorised the training set, how would the
nearest-neighbour distances of generated images compare with those of
validation images?

_Your prediction:_ …

In [ ]:
# Supplied diagnostics.
def bright_area(images):
    return (images > 0).float().mean(dim=(1, 2, 3))  # fraction of pixels brighter than mid-gray

def nearest_training_image(images):
    distances = torch.cdist(images.flatten(1), train.flatten(1))  # Euclidean, over 256 pixels
    return distances.min(dim=1)  # distance to, and index of, the nearest training image

if samples is not None:
    generated_distance, generated_index = nearest_training_image(samples)
    validation_distance, _ = nearest_training_image(valid)
    print("Distance to the nearest training image")
    for name, d in [("Generated", generated_distance), ("Validation", validation_distance)]:
        print(f"  {name:10s} smallest {d.min().item():.2f} | median {d.median().item():.2f} | "
              f"largest {d.max().item():.2f}")
    closest = generated_distance.argsort()[:8]
    show_rows([samples[closest], train[generated_index[closest]]],
              labels=["Generated", "Nearest\ntraining"],
              titles=[f"{generated_distance[i].item():.2f}" for i in closest],
              suptitle="The eight generated images closest to a training image (distance above)")

    groups = [("Validation images", valid, GRAY),
              (f"Generated, {first_updates} updates", first_samples, BLUE),
              (f"Generated, {len(losses)} updates", samples, GREEN)]
    fig, axes = plt.subplots(len(groups), 1, figsize=(6, 4.6), sharex=True, sharey=True)
    for ax, (name, images, color) in zip(axes, groups):
        area = bright_area(images)
        empty = (area == 0).sum().item()
        ax.hist(area.clamp(max=0.399).numpy(), bins=np.linspace(0, 0.4, 21), color=color,
                rwidth=0.85)
        ax.set_title(f"{name}: mean {area.mean().item():.3f}, {empty} of {len(area)} empty",
                     fontsize=9, loc="left")
        ax.set_ylabel("Images")
        ax.grid(axis="y", color="#e6e6e6", linewidth=0.8)
        ax.set_axisbelow(True)
    axes[-1].set_xlabel("Bright area (fraction of pixels above 0)")
    fig.tight_layout()
    plt.show()
else:
    print("Train the network for the full 1800 updates first, then rerun in order.")

**Record and explain:**

1. Are the generated disks copies of training images? Compare the two distance
   summaries and look at the closest pairs.
2. Does the generated size distribution match the validation distribution?
   Compare with another pair: the direction of a mismatch can differ between
   computers. What does that say about judging a generator from one training run?
3. Every check is blind to some failures. Name one failure that the
   nearest-neighbour check would miss, and one that the bright-area check would
   miss. Which experiment today produced perfect-looking images that fail as a
   *set*?

_Your observations:_ …

**Individual exit ticket:** in four sentences, explain (1) what the network
receives and what it predicts, (2) where its training target comes from,
(3) why it is told the timestep, and (4) what one reverse step does with the
prediction.

_Your exit ticket:_ …

## 8. Extended exercise: fill in a hidden region

A trained denoiser is a reusable model of "what disks look like". Use it to
complete an image of which only some pixels are known. This is **inpainting**.
The network is not retrained.

The idea: run the usual reverse process, but after every step **overwrite the
known pixels**. After the step from $t$ the state is $x_{t-1}$, so the known
pixels are replaced by a copy of the known image noised to level $t-1$. The
hidden pixels keep the sampler's values. With a mask $m$ that is 1 for known
and 0 for hidden pixels:

$$x_{t-1}\;\leftarrow\;m\odot\underbrace{\big(\sqrt{\bar\alpha_{t-1}}\,x^{\text{known}}+\sqrt{1-\bar\alpha_{t-1}}\,\epsilon\big)}_{\text{known image, noised to level }t-1}\;+\;(1-m)\odot x_{t-1}.$$

**Predict:** only the left half of each image is known. (a) If it shows part of
a disk, will different runs agree on the hidden half? (b) If it shows almost
nothing of the disk, what must the model invent?

_Your prediction:_ …

In [ ]:
@torch.no_grad()
def inpaint(model, known, mask, schedule, seed):
    # Generate the pixels where `mask` is 0. Pixels where `mask` is 1 are taken from `known`.
    model.eval()
    generator = torch.Generator().manual_seed(seed)
    x = torch.randn(known.shape, generator=generator)
    for step in range(schedule.steps, 0, -1):
        z = torch.randn(x.shape, generator=generator)
        x = ddpm_step(model, x, step, z, schedule)  # x is now x_{t-1}
        earlier = torch.full((len(x),), step - 1)
        noise = torch.randn(x.shape, generator=generator)
        known_noisy = q_sample(known, earlier, noise, schedule)  # known image at noise level t-1
        # TODO 6: keep known_noisy where the mask is 1 and x where the mask is 0.
        x = x
    return x

if samples is not None:
    mask = torch.zeros(1, 1, 16, 16)
    mask[..., :8] = 1  # the left half is known
    visible = ((valid + 1) / 2 * mask).sum(dim=(1, 2, 3))  # how much of each disk can be seen
    # Three validation images: the largest, a medium and the smallest visible part of the disk.
    medium = (visible - visible.median()).abs().argmin().item()
    cases = [visible.argmax().item(), medium, visible.argmin().item()]
    rows, inpaint_ready = [], True
    for case in cases:
        truth = valid[case:case + 1]
        known = (truth * mask).repeat(16, 1, 1, 1)  # the model never sees the hidden pixels
        completions = inpaint(model, known, mask, schedule, seed=41)  # 16 runs, different noise
        hidden = completions * (1 - mask)
        inpaint_ready = (inpaint_ready and torch.allclose(completions * mask, known, atol=1e-5)
                         and not torch.allclose(hidden, torch.zeros_like(hidden), atol=1e-2))
        disagreement = 2 * completions.std(dim=0, keepdim=True) - 1  # black: all 16 runs agree
        rows.append(torch.cat([truth, known[:1], completions[:6], disagreement]))
    if inpaint_ready:
        runs = [f"Run {i}" for i in range(1, 7)]
        show_rows(rows, labels=["Mostly\nvisible", "Partly\nvisible", "Barely\nvisible"],
                  titles=["True image", "Known part", *runs, "Disagreement"])
    else:
        print("The known pixels are not preserved, or nothing was generated.")
        print("Complete TODO 6, then rerun this cell.")
else:
    print("Train the network for the full 1800 updates first, then rerun in order.")

The last column shows the pixelwise standard deviation over 16 runs: black
where all runs agree, bright where they differ.

**Inspect and discuss:**

1. In which case do the runs agree, in which do they differ? Where in the image
   is the disagreement concentrated?
2. Every completion looks convincing, yet most differ from the true image.
   Where would that be a problem if these were medical scans with a hidden
   region? What could you show a user in addition to a single completed image?
3. We changed only the sampling loop, not the network. Text-to-image models
   steer the same kind of sampler with a text prompt instead of known pixels.
   Which additional input would the network need for that?

_Your observations:_ …

### Where next

- **Lab 7** in the course book asks how many of the 64 network evaluations a
  sample really needs, and introduces the deterministic DDIM sampler.
- **Lesson 7** connects today's parts to conditioning, guidance and latent
  diffusion in current image generators.

### References

- [Course book, Lesson 7: Diffusion Image Generation](https://healthml.github.io/deep-learning-book/chapters/07-diffusion/concepts.html) and [Lab 7](https://healthml.github.io/deep-learning-book/chapters/07-diffusion/lab.html): notation, the cosine schedule with 64 steps, and the time-conditioned U-Net used here.
- [Ho, Jain & Abbeel (2020), Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239): noise-prediction loss and the DDPM sampler.
- [Nichol & Dhariwal (2021), Improved Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2102.09672): the cosine schedule.
- [Lugmayr et al. (2022), RePaint](https://arxiv.org/abs/2201.09865): inpainting by overwriting the known pixels during sampling.

Original classroom exercise with synthetic data. Repository licenses: code MIT;
original prose and exercises CC BY-SA 4.0.